# Day 18 — Categorical variables, scaling and preprocessing pipelines

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Preprocessing pipelines

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-018/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd
import numpy as np

# A small, messy dataset: numbers and categories, with gaps in both
df = pd.DataFrame({
    "study_hours": [3, 5, np.nan, 4, 6, 2],
    "attendance": [75, 90, 65, np.nan, 85, 55],
    "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad", "Mumbai", np.nan],
    "grade": ["A", "B", "A", "C", np.nan, "B"],
    "result": ["Pass", "Pass", "Fail", "Pass", "Pass", "Fail"]})
print(df)

# Each column's kind and gaps (df.info() shows the same, in a format that differs between pandas versions)
kind = ["number" if pd.api.types.is_numeric_dtype(df[c]) else "text" for c in df.columns]
print(pd.DataFrame({"kind": kind, "non-null": df.notna().sum(), "missing": df.isnull().sum()}).to_string())

   study_hours  attendance       city grade result
0          3.0        75.0  Hyderabad     A   Pass
1          5.0        90.0     Mumbai     B   Pass
2          NaN        65.0      Delhi     A   Fail
3          4.0         NaN  Hyderabad     C   Pass
4          6.0        85.0     Mumbai   NaN   Pass
5          2.0        55.0        NaN     B   Fail
               kind  non-null  missing
study_hours  number         5        1
attendance   number         5        1
city           text         5        1
grade          text         5        1
result         text         6        0


In [ ]:
# Feature groups: each group gets its own preprocessing; the target is kept apart from the inputs
numeric_features = ["study_hours", "attendance"]
categorical_features = ["city", "grade"]
target = "result"
X = df[numeric_features + categorical_features]
y = df[target]
print("Numerical:  ", numeric_features)
print("Categorical:", categorical_features)
print("X:", X.shape, "| y:", y.shape)

Numerical:   ['study_hours', 'attendance']
Categorical: ['city', 'grade']
X: (6, 4) | y: (6,)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])

# What each step learned from these six students
Xt = preprocessor.fit_transform(X)
num = preprocessor.named_transformers_["numeric"]
cat = preprocessor.named_transformers_["categorical"]
print("Medians that fill gaps:   ", num.named_steps["imputer"].statistics_)
print("Means used for scaling:   ", num.named_steps["scaler"].mean_.round(2))
print("Most frequent (fill gaps):", cat.named_steps["imputer"].statistics_)
print("Output shape:", Xt.shape)
cols = [n.split("__")[1] for n in preprocessor.get_feature_names_out()]
print("Columns:", ", ".join(cols[:2]), "+", len(cols) - 2, "0/1 columns:", ", ".join(c.split("_")[1] for c in cols[2:]))

Medians that fill gaps:    [ 4. 75.]
Means used for scaling:    [ 4.   74.17]
Most frequent (fill gaps): ['Hyderabad' 'A']
Output shape: (6, 8)
Columns: study_hours, attendance + 6 0/1 columns: Delhi, Hyderabad, Mumbai, A, B, C


In [ ]:
# The six students after preprocessing: no gaps, numbers scaled, categories as 0/1 columns
def show(rows):
    """Scaled numbers to 2 decimals, the 0/1 category columns as whole numbers, with short headers."""
    short = ["hours", "attend", "Delhi", "Hyderabad", "Mumbai", "grade_A", "grade_B", "grade_C"]
    out = pd.DataFrame(rows.toarray() if hasattr(rows, "toarray") else rows, columns=short).round(2)
    out[short[2:]] = out[short[2:]].astype(int)
    print(out.to_string())

show(Xt)

   hours  attend  Delhi  Hyderabad  Mumbai  grade_A  grade_B  grade_C
0  -0.77    0.07      0          1       0        1        0        0
1   0.77    1.35      0          0       1        0        1        0
2   0.00   -0.78      1          0       0        1        0        0
3   0.00    0.07      0          1       0        0        0        1
4   1.55    0.93      0          0       1        1        0        0
5  -1.55   -1.64      0          1       0        0        1        0


In [ ]:
# handle_unknown="ignore": a city never seen in training becomes all zeros instead of an error
new_student = pd.DataFrame({"study_hours": [5], "attendance": [80], "city": ["Bengaluru"], "grade": ["B"]})
show(preprocessor.transform(new_student))

strict = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                                      ("encoder", OneHotEncoder())]), categorical_features)])
strict.fit(X)
try:
    strict.transform(new_student)
except ValueError as error:
    print("Without it:", str(error).split(" during")[0])

   hours  attend  Delhi  Hyderabad  Mumbai  grade_A  grade_B  grade_C
0   0.77     0.5      0          0       0        0        1        0
Without it: Found unknown categories ['Bengaluru'] in column 0


In [ ]:
from sklearn.model_selection import train_test_split

# The same preprocessor on our 150 students. Class is the categorical column; "Unknown" means missing
students = pd.read_csv("student_data_150.csv")
students["Class"] = students["Class"].replace("Unknown", np.nan)
features_num = ["StudyHours", "Attendance", "PreviousScore", "Age"]
features_cat = ["Class"]
X_all, y_all = students[features_num + features_cat], students["Passed"]
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.15, random_state=42, stratify=y_all)
print("Missing values per column:", X_all.isnull().sum().to_dict())
print(f"Train {X_train.shape} | test {X_test.shape}")

real = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), features_num),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="ignore"))]), features_cat)])
X_train_p = real.fit_transform(X_train)          # learn medians, means, spreads and categories from training only
X_test_p = real.transform(X_test)                # apply them unchanged to the test students
print("Processed:", X_train_p.shape, X_test_p.shape)
print("Columns:  ", [n.split("__")[1] for n in real.get_feature_names_out()])
print("Class filled with:", real.named_transformers_["categorical"].named_steps["imputer"].statistics_[0])

Missing values per column: {'StudyHours': 0, 'Attendance': 0, 'PreviousScore': 0, 'Age': 0, 'Class': 5}
Train (127, 5) | test (23, 5)
Processed: (127, 7) (23, 7)
Columns:   ['StudyHours', 'Attendance', 'PreviousScore', 'Age', 'Class_A', 'Class_B', 'Class_C']
Class filled with: A


In [ ]:
# The leak: fit on all 150 students first and the test students shape the statistics the model is scored with
leaky = ColumnTransformer([("numeric", StandardScaler(), features_num)]).fit(X_all)
safe = ColumnTransformer([("numeric", StandardScaler(), features_num)]).fit(X_train)
compare = pd.DataFrame({"mean (train only)": safe.named_transformers_["numeric"].mean_,
                        "mean (all 150)": leaky.named_transformers_["numeric"].mean_}, index=features_num).round(3)
print(compare.to_string())
print("Test students scaled differently:", int((np.abs(leaky.transform(X_test) - safe.transform(X_test)) > 1e-9).any(axis=1).sum()),
      "of", len(X_test))

               mean (train only)  mean (all 150)
StudyHours                 3.614           3.573
Attendance                79.126          79.487
PreviousScore             62.394          62.660
Age                       11.031          11.020
Test students scaled differently: 23 of 23


## 2. Advanced categorical encoding

In [ ]:
# One-hot with an unseen category: categories are sorted alphabetically (Delhi, Hyderabad, Mumbai)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)   # older scikit-learn: sparse=False
encoder.fit([["Hyderabad"], ["Mumbai"], ["Delhi"]])
print(encoder.categories_[0])
print(encoder.transform([["Hyderabad"], ["Bengaluru"]]))

['Delhi' 'Hyderabad' 'Mumbai']
[[0. 1. 0.]
 [0. 0. 0.]]


In [ ]:
# Frequency encoding: learn the counts from training data, with a fallback for cities never seen
train_cities = pd.DataFrame({"city": ["Hyderabad", "Mumbai", "Hyderabad", "Delhi", "Hyderabad", "Mumbai"]})
frequency_map = train_cities["city"].value_counts().to_dict()
print(frequency_map)
train_cities["city_frequency"] = train_cities["city"].map(frequency_map)
print(train_cities.to_string(index=False))
new_cities = pd.Series(["Mumbai", "Bengaluru"])
print(new_cities.map(frequency_map).fillna(0).astype(int).tolist(), "<- Bengaluru gets the fallback 0")

{'Hyderabad': 3, 'Mumbai': 2, 'Delhi': 1}
     city  city_frequency
Hyderabad               3
   Mumbai               2
Hyderabad               3
    Delhi               1
Hyderabad               3
   Mumbai               2
[2, 0] <- Bengaluru gets the fallback 0


In [ ]:
from sklearn.preprocessing import TargetEncoder

# Target encoding: each city's purchase rate, and what scikit-learn's TargetEncoder makes of it
buyers = pd.DataFrame({"city": ["Hyderabad", "Hyderabad", "Mumbai", "Mumbai", "Delhi"], "purchase": [1, 1, 0, 1, 0]})
print("Raw purchase rate:", buyers.groupby("city")["purchase"].mean().to_dict())
te = TargetEncoder(target_type="binary", smooth="auto").fit(buyers[["city"]], buyers["purchase"])
print("Overall rate:     ", buyers["purchase"].mean())
print("Smoothed:         ", {c: round(float(v), 2) for c, v in zip(te.categories_[0], te.encodings_[0])})

Raw purchase rate: {'Delhi': 0.0, 'Hyderabad': 1.0, 'Mumbai': 0.5}
Overall rate:      0.6
Smoothed:          {'Delhi': 0.0, 'Hyderabad': 1.0, 'Mumbai': 0.53}


In [ ]:
# Rare categories: learn which are common from training data, group the rest as "Other"
pay = pd.DataFrame({"payment_method": ["UPI", "UPI", "UPI", "Card", "Card", "Wallet", "Other"]})
counts = pay["payment_method"].value_counts()
common_categories = counts[counts >= 2].index
pay["payment_method_grouped"] = pay["payment_method"].where(pay["payment_method"].isin(common_categories), "Other")
print(counts.to_dict())
print(pay.to_string(index=False))

{'UPI': 3, 'Card': 2, 'Wallet': 1, 'Other': 1}
payment_method payment_method_grouped
           UPI                    UPI
           UPI                    UPI
           UPI                    UPI
          Card                   Card
          Card                   Card
        Wallet                  Other
         Other                  Other


In [ ]:
# Our 150 students: how many distinct values does each column have?
print(students.nunique().to_string())
# One-hot on the student ID would add one column per training student, none of them useful for a new student
id_onehot = OneHotEncoder(handle_unknown="ignore").fit(students.loc[X_train.index, ["Student"]])
print("One-hot columns for Student on the training set:", len(id_onehot.categories_[0]))

Student          150
Class              3
Age                3
StudyHours        14
Attendance        42
PreviousScore     57
ExamScore         51
Passed             2
Source             2
One-hot columns for Student on the training set: 127


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Target-encoding leakage with an identifier. Naive: each student's "pass rate" computed on all training rows,
# which is their own result, then cross-validated. Safe: TargetEncoder inside the pipeline (out-of-fold on fit)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train = students.loc[X_train.index]
y = train["Passed"].astype(int)
numbers = ["StudyHours", "Attendance", "PreviousScore"]
lr = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
naive = train[numbers].assign(StudentRate=y.groupby(train["Student"]).transform("mean"))
safe = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), numbers),
                                             ("id", TargetEncoder(target_type="binary"), ["Student"])])),
                 ("model", LogisticRegression(max_iter=1000))])
print(f"Numbers only:                       CV accuracy {cross_val_score(lr, train[numbers], y, cv=cv).mean():.3f}")
print(f"+ student pass rate (naive, leaks): CV accuracy {cross_val_score(lr, naive, y, cv=cv).mean():.3f}")
print(f"+ TargetEncoder in the pipeline:    CV accuracy {cross_val_score(safe, train[numbers + ['Student']], y, cv=cv).mean():.3f}")

Numbers only:                       CV accuracy 0.796
+ student pass rate (naive, leaks): CV accuracy 1.000


+ TargetEncoder in the pipeline:    CV accuracy 0.796


## 3. Scaling techniques

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Three scalers on five students; each column is scaled on its own
sample = pd.DataFrame({"study_hours": [2, 4, 6, 8, 10], "attendance": [50, 60, 75, 85, 95],
                       "family_income": [20000, 30000, 45000, 60000, 90000]})
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    scaled = pd.DataFrame(scaler.fit_transform(sample), columns=sample.columns)
    print(f"\n{type(scaler).__name__}")
    print(scaled.round(2).to_string(index=False))
standardized = StandardScaler().fit_transform(sample)
print("\nStandardScaler means:", standardized.mean(axis=0).round(2) + 0, "| standard deviations:", standardized.std(axis=0).round(2))


StandardScaler
 study_hours  attendance  family_income
       -1.41       -1.41          -1.18
       -0.71       -0.80          -0.77
        0.00        0.12          -0.16
        0.71        0.74           0.45
        1.41        1.35           1.67

MinMaxScaler
 study_hours  attendance  family_income
        0.00        0.00           0.00
        0.25        0.22           0.14
        0.50        0.56           0.36
        0.75        0.78           0.57
        1.00        1.00           1.00

RobustScaler
 study_hours  attendance  family_income
        -1.0        -1.0          -0.83
        -0.5        -0.6          -0.50
         0.0         0.0           0.00
         0.5         0.4           0.50
         1.0         0.8           1.50

StandardScaler means: [0. 0. 0.] | standard deviations: [1. 1. 1.]


In [ ]:
# Transaction amounts with one extreme value
values = np.array([[100], [200], [300], [400], [50000]])
print("mean", values.mean(), "| std", values.std().round(1), "| median", np.median(values),
      "| IQR", np.percentile(values, 75) - np.percentile(values, 25))
for name, scaler in {"Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler()}.items():
    print(f"{name:<9}", scaler.fit_transform(values).ravel().round(3))

mean 10200.0 | std 19900.3 | median 300.0 | IQR 200.0
Standard  [-0.508 -0.503 -0.497 -0.492  2.   ]
MinMax    [0.    0.002 0.004 0.006 1.   ]
Robust    [ -1.   -0.5   0.    0.5 248.5]


In [ ]:
# MinMaxScaler maps the *training* range to 0-1; a new value outside it goes past 1 unless clip=True
hours_train = X_train[["StudyHours"]]
print("Training study hours:", hours_train["StudyHours"].min(), "to", hours_train["StudyHours"].max())
new_hours = pd.DataFrame({"StudyHours": [3.0, 9.0]})
print("MinMaxScaler():         ", MinMaxScaler().fit(hours_train).transform(new_hours).ravel().round(2))
print("MinMaxScaler(clip=True):", MinMaxScaler(clip=True).fit(hours_train).transform(new_hours).ravel().round(2))

Training study hours: 0.5 to 7.0
MinMaxScaler():          [0.38 1.31]
MinMaxScaler(clip=True): [0.38 1.  ]


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# Choose the scaler like any other setting: by cross-validation on the training students only.
# "passthrough" means no scaling at all
X_num = train[["StudyHours", "Attendance", "PreviousScore"]]
scalers = [StandardScaler(), MinMaxScaler(), RobustScaler(), "passthrough"]
results = {}
for name, clf in [("Logistic regression", LogisticRegression(max_iter=1000)), ("KNN (k=5)", KNeighborsClassifier())]:
    search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("classifier", clf)]),
                          {"scaler": scalers}, cv=cv).fit(X_num, y)
    results[name] = search.cv_results_["mean_test_score"].round(3)
names = [type(sc).__name__ if sc != "passthrough" else "none (unscaled)" for sc in scalers]
print("CV accuracy on the 127 training students")
print(pd.DataFrame(results, index=names).to_string())

CV accuracy on the 127 training students
                 Logistic regression  KNN (k=5)
StandardScaler                 0.796      0.795
MinMaxScaler                   0.788      0.787
RobustScaler                   0.796      0.780
none (unscaled)                0.796      0.772


## 4. ColumnTransformer in practice

In [ ]:
# A tiny banking dataset with gaps in both numeric columns; the target is loan_approved
loans = pd.DataFrame({
    "age": [25, 32, 45, 28, np.nan, 38],
    "income": [45000, 75000, 90000, 55000, 65000, np.nan],
    "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad", "Mumbai", "Delhi"],
    "employment": ["Salaried", "Business", "Salaried", "Self-employed", "Salaried", "Business"],
    "loan_approved": [1, 1, 1, 0, 1, 0]})
print(loans)
print(loans.isnull().sum().to_dict())

    age   income       city     employment  loan_approved
0  25.0  45000.0  Hyderabad       Salaried              1
1  32.0  75000.0     Mumbai       Business              1
2  45.0  90000.0      Delhi       Salaried              1
3  28.0  55000.0  Hyderabad  Self-employed              0
4   NaN  65000.0     Mumbai       Salaried              1
5  38.0      NaN      Delhi       Business              0
{'age': 1, 'income': 1, 'city': 0, 'employment': 0, 'loan_approved': 0}


In [ ]:
# Features and target, column groups, and the ColumnTransformer: (name, pipeline, columns) for each group
X = loans.drop(columns=["loan_approved"])
y_loan = loans["loan_approved"]
numeric_features, categorical_features = ["age", "income"], ["city", "employment"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])
X_tr, X_te, y_tr, y_te = train_test_split(X, y_loan, test_size=0.2, random_state=42, stratify=y_loan)
print("Training rows:", list(X_tr.index), "| test rows:", list(X_te.index))
X_tr_p = preprocessor.fit_transform(X_tr)      # learn on the 4 training rows
X_te_p = preprocessor.transform(X_te)          # apply to the 2 test rows
print("Training shape:", X_tr_p.shape, "| testing shape:", X_te_p.shape)

Training rows: [1, 0, 5, 2] | test rows: [4, 3]
Training shape: (4, 7) | testing shape: (2, 7)


In [ ]:
# The transformed columns, named by step: they come from the categories in the *training* rows
for name in preprocessor.get_feature_names_out():
    print(name)
print("Cities seen in training:", list(preprocessor.named_transformers_["categorical"]
                                       .named_steps["encoder"].categories_[0]))

numeric__age
numeric__income
categorical__city_Delhi
categorical__city_Hyderabad
categorical__city_Mumbai
categorical__employment_Business
categorical__employment_Salaried
Cities seen in training: ['Delhi', 'Hyderabad', 'Mumbai']


In [ ]:
# Preprocessing + model in one Pipeline: raw rows in, predictions out. Six rows can't measure anything:
# this shows the mechanics only
loan_model = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(max_iter=1000))])
loan_model.fit(X_tr, y_tr)
print("Test predictions:", loan_model.predict(X_te), "| actual:", y_te.to_numpy())
new_customer = pd.DataFrame({"age": [35], "income": [80000], "city": ["Bengaluru"], "employment": ["Salaried"]})
print("Bengaluru customer: prediction", loan_model.predict(new_customer)[0],
      "| P(approved)", loan_model.predict_proba(new_customer)[0, 1].round(2))

Test predictions: [1 1] | actual: [1 0]
Bengaluru customer: prediction 1 | P(approved) 0.84


In [ ]:
# remainder: columns not listed are dropped by default; "passthrough" keeps them unchanged
extra = X_tr.assign(branch_code=[101, 102, 103, 104])
dropped = ColumnTransformer([("numeric", numeric_pipeline, numeric_features)], verbose_feature_names_out=False)
kept = ColumnTransformer([("numeric", numeric_pipeline, numeric_features)], remainder="passthrough",
                         verbose_feature_names_out=False)
print("Default:    ", list(dropped.fit(extra).get_feature_names_out()))
print("passthrough:", list(kept.fit(extra).get_feature_names_out()))

Default:     ['age', 'income']
passthrough: ['age', 'income', 'city', 'employment', 'branch_code']


In [ ]:
from sklearn.compose import make_column_selector

# On our students: pick the column groups by type, so a new numeric column is handled without editing lists
student_X = students.loc[X_train.index, ["StudyHours", "Attendance", "PreviousScore", "Age", "Class"]]
by_type = ColumnTransformer([("numeric", numeric_pipeline, make_column_selector(dtype_include="number")),
                             ("categorical", categorical_pipeline, make_column_selector(dtype_exclude="number"))],
                            verbose_feature_names_out=False)                # short names: no "numeric__" prefix
student_model = Pipeline([("preprocessor", by_type), ("classifier", LogisticRegression(max_iter=1000))])
print(f"CV accuracy: {cross_val_score(student_model, student_X, y, cv=cv).mean():.3f}")
student_model.fit(student_X, y)
print("Columns:", list(student_model.named_steps["preprocessor"].get_feature_names_out()))
twins = pd.DataFrame({"StudyHours": [3, 3], "Attendance": [80, 80], "PreviousScore": [55, 55], "Age": [11, 11],
                      "Class": ["B", "D"]})
print("Same student, P(pass) in class B, unseen class D:", student_model.predict_proba(twins)[:, 1].round(3))

CV accuracy: 0.780
Columns: ['StudyHours', 'Attendance', 'PreviousScore', 'Age', 'Class_A', 'Class_B', 'Class_C']
Same student, P(pass) in class B, unseen class D: [0.699 0.633]


## 5. End-to-end pipeline

In [ ]:
# Step 1: eight loan applicants (no gaps this time; the pipeline still imputes, ready for real data)
bank = pd.DataFrame({
    "age": [25, 32, 45, 28, 38, 50, 29, 41],
    "income": [45000, 75000, 90000, 55000, 65000, 100000, 48000, 85000],
    "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad", "Mumbai", "Delhi", "Bengaluru", "Mumbai"],
    "employment": ["Salaried", "Business", "Salaried", "Self-employed", "Salaried", "Business", "Salaried", "Business"],
    "loan_approved": [1, 1, 1, 0, 1, 1, 0, 1]})
print(bank.head())
print("Missing:", bank.isnull().sum().to_dict())

   age  income       city     employment  loan_approved
0   25   45000  Hyderabad       Salaried              1
1   32   75000     Mumbai       Business              1
2   45   90000      Delhi       Salaried              1
3   28   55000  Hyderabad  Self-employed              0
4   38   65000     Mumbai       Salaried              1
Missing: {'age': 0, 'income': 0, 'city': 0, 'employment': 0, 'loan_approved': 0}


In [ ]:
# Steps 2-7: features and target, the two pipelines, ColumnTransformer, the complete model, the split, training
X_bank, y_bank = bank.drop(columns=["loan_approved"]), bank["loan_approved"]
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), ["age", "income"]),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="ignore"))]), ["city", "employment"])])
loan_model = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(max_iter=1000))])
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bank, y_bank, test_size=0.25, random_state=42, stratify=y_bank)
loan_model.fit(Xb_train, yb_train)              # raw DataFrame in: imputing, scaling and encoding happen inside
predictions = loan_model.predict(Xb_test)
print("Train:", len(Xb_train), "rows | test:", len(Xb_test), "rows", list(Xb_test.index))
print("Predictions:", predictions, "| actual:", yb_test.to_numpy())

Train: 6 rows | test: 2 rows [6, 5]
Predictions: [1 1] | actual: [0 1]


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

# Step 8: evaluation. Two test applicants make every metric a coin's worth of evidence: mechanics only
print("Accuracy: ", accuracy_score(yb_test, predictions))
print("Precision:", precision_score(yb_test, predictions, zero_division=0))
print("Recall:   ", recall_score(yb_test, predictions, zero_division=0))
print("F1:       ", round(f1_score(yb_test, predictions, zero_division=0), 3))
print(confusion_matrix(yb_test, predictions, labels=[0, 1]))

Accuracy:  0.5
Precision: 0.5
Recall:    1.0
F1:        0.667
[[0 1]
 [0 1]]


In [ ]:
import warnings

# Step 9: 3-fold cross-validation on the 6 training applicants, as the example does. Only one is unapproved
print("Unapproved in training:", int((yb_train == 0).sum()), "of", len(yb_train))
with warnings.catch_warnings(record=True) as caught:                # collect the warnings to list them below
    warnings.simplefilter("always")
    for k, (fit_rows, score_rows) in enumerate(StratifiedKFold(n_splits=3).split(Xb_train, yb_train), 1):
        print(f"Fold {k}: fits on outcomes {sorted(set(yb_train.iloc[fit_rows]))}, scores on {list(yb_train.iloc[score_rows])}")
    scores = cross_val_score(loan_model, Xb_train, yb_train, cv=3, scoring="f1")
print("Fold F1 scores:", scores.round(2), "| mean", scores.mean().round(2))
print("Warnings:", sorted({type(w.message).__name__ for w in caught}))

Unapproved in training: 1 of 6
Fold 1: fits on outcomes [0, 1], scores on [1, 1]
Fold 2: fits on outcomes [0, 1], scores on [1, 1]
Fold 3: fits on outcomes [1], scores on [0, 1]


Fold F1 scores: [0.67 1.    nan] | mean nan
Warnings: ['FitFailedWarning', 'UserWarning']


In [ ]:
import joblib

# Steps 10-11: save the whole fitted pipeline (preprocessing + model), load it, predict for an unseen city.
# joblib runs Python code when loading: only load files you created or trust
joblib.dump(loan_model, "loan_approval_pipeline.joblib")
loaded_model = joblib.load("loan_approval_pipeline.joblib")
new_customer = pd.DataFrame({"age": [35], "income": [80000], "city": ["Bengaluru"], "employment": ["Salaried"]})
print("Cities in training:", sorted(Xb_train["city"].unique()))          # the only Bengaluru applicant is in test
print("Prediction:", loaded_model.predict(new_customer)[0])
print("Probabilities [not approved, approved]:", loaded_model.predict_proba(new_customer)[0].round(2))

Cities in training: ['Delhi', 'Hyderabad', 'Mumbai']


Prediction: 1
Probabilities [not approved, approved]: [0.09 0.91]


In [ ]:
# The same workflow on our 127 training students: cross-validate the complete pipeline, fit, save, reload,
# and check the reloaded pipeline predicts exactly what the original did. (The 23 test students wait for Section 6.)
student_cols = ["StudyHours", "Attendance", "PreviousScore", "Age", "Class"]
pass_model = Pipeline([("preprocessor", ColumnTransformer([
                          ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                           ["StudyHours", "Attendance", "PreviousScore", "Age"]),
                          ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                                    ("encoder", OneHotEncoder(handle_unknown="ignore"))]), ["Class"])])),
                       ("classifier", LogisticRegression(max_iter=1000))])
fail_f1 = make_scorer(f1_score, pos_label=0)
scores = cross_val_score(pass_model, student_X[student_cols], y, cv=cv, scoring=fail_f1)
print("5-fold F1 for fail:", scores.round(3), "| mean", scores.mean().round(3))
pass_model.fit(student_X[student_cols], y)
joblib.dump(pass_model, "student_pass_pipeline.joblib")
reloaded = joblib.load("student_pass_pipeline.joblib")
same = (reloaded.predict_proba(student_X[student_cols]) == pass_model.predict_proba(student_X[student_cols])).all()
print("Reloaded pipeline gives identical probabilities for all 127:", bool(same))

5-fold F1 for fail: [0.588 0.625 0.364 0.6   0.714] | mean 0.578
Reloaded pipeline gives identical probabilities for all 127: True


In [ ]:
# Production: validate an incoming record before it reaches the pipeline (a FastAPI service would do this per request)
RULES = {"StudyHours": (0, 24), "Attendance": (0, 100), "PreviousScore": (0, 100), "Age": (5, 25)}

def validate(record: dict) -> list[str]:
    problems = [f"missing {c}" for c in student_cols if c not in record]
    for col, (low, high) in RULES.items():
        value = record.get(col)
        if value is not None and not (isinstance(value, (int, float)) and low <= value <= high):
            problems.append(f"{col}={value!r} outside {low}-{high}")
    return problems

for record in [{"StudyHours": 3, "Attendance": 80, "PreviousScore": 55, "Age": 11, "Class": "B"},
               {"StudyHours": 3, "Attendance": 140, "PreviousScore": 55, "Class": "B"}]:
    problems = validate(record)
    print("rejected:" if problems else "accepted:", problems or reloaded.predict_proba(pd.DataFrame([record]))[0, 1].round(3))

accepted: 0.699
rejected: ['missing Age', 'Attendance=140 outside 0-100']
